# Modeling Sequential Data Using Recurrent Neural Networks
**Modeling sequential data with recurrent neural networks**

In the previous chapter, we studied `CNNs`, from basic building blocks to implementing `CNNs` with `tensorflow`. In this chapter, we will explore `RNNs` (Recurrent Neural Networks) and their variants for sequential data.

1. Introduction to sequential data and common problems.
2. Recurrent models and their variants.
3. Backpropagation Through Time (BPTT) and the vanishing gradient problem.
4. Truncated Backpropagation Through Time (TBPTT).
5. Implementing `RNNs` and their variants using `numpy`.
6. Attention and the `transformer` model (self-attention mechanism) [*].

## Introduction to sequential data and common problems

### Issues with sequential data in memoryless `NN` models

We begin with the `slot filling` task and knowledge from the previous chapter. Suppose we have a real-world problem as follows:

An airline wants to build an automated system to extract flight information from customer messages sent to their consulting system and suggest appropriate flight schedules based on the customer’s needs. The business process is illustrated below.

![figure 1](images/RNN/1.png)

`slots` are the fields that need to be filled based on the information provided by the passenger. In the above example, we define two `slots`: *time of arrival* and *destination*. Our task is to fill in the missing information for these fields, which is known as `slot filling`.

The question is whether we can solve this problem using standard ANNs.

To solve this problem using `ANNs`, we feed each word in the sentence one by one into the ANN and predict which `slot` that word most likely belongs to.

![figure 2](images/RNN/2.png)

For example, when we input the word **Taipei** into the ANN model, the output may indicate that **Taipei** has a high probability of belonging to the `destination` slot. Let us consider a more challenging scenario.

Now suppose we have three `slots`: *destination*, *place of departure*, and *time of arrival*. Consider the sentence:

**I would like to leave Taipei on November 2nd**.

![figure 3](images/RNN/3.png)

At this point, a problem arises with standard `ANNs`. The model cannot determine whether **Taipei** is a *destination* or a *place of departure*. This ambiguity arises because `ANNs` do not retain memory of previously processed words—specifically, they cannot consider context words such as **leave** or **arrive**. In some cases, **Taipei** may not belong to any `slot` at all.

Therefore, we can conclude that a key characteristic of sequential data is **order dependence**, and to model such data effectively, **we need to build models that have memory**.

### The problem of sequential representation

We now understand the challenges of handling sequential data with `ANNs` and the need for models capable of handling sequence order. But how do we represent such ordered data in machine learning models?

Returning to the problem in **Section 1.1**, suppose we have $N$ sentences corresponding to $N$ training samples. Assume that all sentences have a similar length of $T$ words. Then each sentence can be represented as a vector of length $T$ (assuming we use tf-idf to convert each word into a real-valued number).

Thus, one data point can be represented as:
$$
\begin{aligned}
\boldsymbol{x} &= \begin{bmatrix}
x^0, x^1, \dots, x^T
\end{bmatrix}
\end{aligned}
$$

Here, the superscript index of each element represents its position in the sentence (row, column). Similarly, the output vector or label is also represented as a vector with $T$ elements. For each index $t$, we have a corresponding input-output pair $(x^t, y^t)$, where $\boldsymbol{y}$ denotes the output vector.

The idea is illustrated in the figure below.

![figure 4](images/RNN/4.png)

### Types of sequential models

Sequential problems have many applications, such as machine translation, image captioning, and text generation. However, different applications correspond to different model architectures. Understanding the differences between these architectures is crucial when approaching a sequential problem.

*See also: [The Unreasonable Effectiveness of Recurrent Neural Networks, by Andrej Karpathy](http://karpathy.github.io/2015/05/21/rnn-effectiveness/).*

![figure 5](images/RNN/5.png)

1. **Many-to-one**: The input is a sequence, but the output is a fixed-size vector. Examples include sentiment analysis, intent detection, and Forex forecasting.

2. **One-to-many**: The input is a fixed-size vector, but the output is a sequence. Example: image captioning.

3. **Many-to-many**: Both input and output are sequences. Examples include machine translation, Named Entity Recognition, Part-of-Speech tagging, and word segmentation.

## Recurrent models and their variants

![figure 0](images/RNN/0.png)

Before diving into implementation, we first discuss typical architectures in recurrent models. This will help us better understand the standard `RNN` and the motivation behind its variants.

### Recurrent Neural Network (RNN)

#### Mechanism of Recurrent Neural Networks

The core mechanism of a basic RNN is recurrence. We can compare a simple RNN with a feedforward network (single hidden-layer ANN) as shown below:

![figure 6](images/RNN/6.png)

Convention: both models have a single hidden layer, but each layer may contain multiple units.

In a `FFN`, information flows in only one direction—from the input to the hidden layer and then to the output. In contrast, in an `RNN`, the hidden layer at time step $t$ receives information from both the input at time $t$ and the hidden layer at time $t-1$.

The flow of information between consecutive time steps allows the `RNN` to retain memory of recent past information. This flow is typically represented as a loop, known as a **recurrent edge**, hence the name `Recurrent Neural Network`.

Like `MLP`, `ANN`, and `CNN`, an `RNN` can have multiple hidden layers, as illustrated below. Note that a single-layer `RNN` is different from a single-layer `MLP` (where a single-layer `MLP` means no hidden layer, e.g., Adaline or Logistic Regression).

![figure 7](images/RNN/7.png)

To better examine the architecture and information flow, we "unroll" the **recurrent edge** as shown in the figure.

Consider a single-layer `RNN`. Similar to `FFN`, the hidden layer receives input from the input layer. However, in an `RNN`, at each time step $t$, the hidden layer receives two inputs:  
(1) the current input $x^t$, and  
(2) the hidden state from the previous time step $h^{t-1}$.

Note: At the initial time step $t=0$, the previous hidden state is usually initialized to zero or a small random value close to zero.

For a multi-layer `RNN`, the information flow can be summarized as:
1. **Layer 1**: The hidden state $h_1^t$ receives input from $x^t$ and the previous hidden state $h_1^{t-1}$.
2. **Layer 2**: The hidden state $h_2^t$ receives input from the output of layer 1 at time $t$, $o_1^t$, and its own previous hidden state $h_2^{t-1}$.

#### Unit activation in RNNs

Now that we understand the architecture and information flow, let us examine how computation and activation occur in the hidden layer. For simplicity, we consider a single hidden layer, but the same concepts apply to multi-layer `RNNs`.

Each directed edge (connection between cells in the unrolled representation) is associated with a weight matrix. These weights are time-invariant, meaning they are shared across all time steps.

The weight matrices in a single-layer `RNN` are shown below:

![figure 8](images/RNN/8.png)

$$
\begin{aligned}
\boldsymbol{a}^{(t)}_h &= \boldsymbol{W}_{xh}\boldsymbol{x}^{(t)} + \boldsymbol{W}_{hh}\boldsymbol{h}^{(t-1)} + \boldsymbol{b}_h \\
\boldsymbol{h}^{(t)} &= \phi_h(\boldsymbol{a}^{(t)}_h) \\
\boldsymbol{o}^{(t)} &= \boldsymbol{W}_{ho}\boldsymbol{h}^{(t)} + \boldsymbol{b}_o
\end{aligned}
$$

#### Backpropagation Through Time (BPTT)

The training method for RNNs was introduced in 1990:  
*Backpropagation Through Time: What It Does and How to Do It* (Paul Werbos, Proceedings of IEEE, 78(10): 1550–1560, 1990).

Computing gradients in RNNs is complex, but the core principles are similar to standard backpropagation. The main idea is that the total loss is the sum of losses over all time steps:

$$
\begin{aligned}
\boldsymbol{L} &= \sum_{t=1}^{T} L^{(t)}
\end{aligned}
$$

Since the loss at time step $t$ depends on all hidden units from time steps $1$ to $t$, the gradient at time $t$ is computed as:

$$
\begin{aligned}
\frac{\partial L^{(t)}}{\partial \boldsymbol{W}_{hh}} &= 
\frac{\partial L^{(t)}}{\partial \boldsymbol{o}^{(t)}} 
\frac{\partial \boldsymbol{o}^{(t)}}{\partial \boldsymbol{h}^{(t)}}
\left( \sum_{k=1}^{t}
\frac{\partial \boldsymbol{h}^{(t)}}{\partial \boldsymbol{h}^{(k)}}
\frac{\partial \boldsymbol{h}^{(k)}}{\partial \boldsymbol{W}_{hh}} \right)
\end{aligned}
$$

Where:
$$
\begin{aligned}
\frac{\partial \boldsymbol{h}^{(t)}}{\partial \boldsymbol{h}^{(k)}} &=
\prod_{i=k+1}^{t} 
\frac{\partial \boldsymbol{h}^{(i)}}{\partial \boldsymbol{h}^{(i-1)}}
\end{aligned}
$$

#### Challenges in learning long-range dependencies

As introduced earlier, `BPTT` is the training method for `RNNs`. However, computing $\frac{\partial \boldsymbol{h}^{(t)}}{\partial \boldsymbol{h}^{(k)}}$ leads to the well-known **vanishing** and **exploding gradient** problems.

![figure 9](images/RNN/9.png)

Essentially, $\frac{\partial \boldsymbol{h}^{(t)}}{\partial \boldsymbol{h}^{(k)}}$ involves multiplying a weight $w$ repeatedly, resulting in $w^{t-k}$. If $|w| < 1$, the product shrinks toward zero as $t-k$ increases, causing vanishing gradients. If $|w| > 1$, the product grows exponentially, causing exploding gradients.

While one might naïvely think setting $|w| = 1$ solves the problem, in practice, standard `RNNs` can only retain information for a limited number of time steps—this is known as **short-term memory**.

Common practical solutions include:
- **Gradient clipping**: limiting the magnitude of gradients.
- **Truncated BPTT (TBPTT)**: limiting how far gradients are propagated backward in time.

#### Long Short-Term Memory (LSTM)

Although gradient clipping and TBPTT help mitigate gradient issues, they may negatively impact learning accuracy. In 1997, Sepp Hochreiter and Jürgen Schmidhuber introduced the `LSTM`, designed to address vanishing and exploding gradients in long-range dependency modeling.

The fundamental building block of an `LSTM` is the **memory cell**, which can be viewed as a replacement for the hidden layer in standard `RNNs`.

![figure 10](images/RNN/10.png)
![figure 11](images/RNN/11.png)

##### Cell state: the core idea of LSTM

Similar to `RNNs`, each `LSTM` memory cell contains a recurrent edge. The value propagated through this edge is called the **cell state**, denoted $\boldsymbol{C}^{(t)}$. The cell state runs horizontally across time steps with linear interactions, allowing information to flow without significant modification.

![figure 12](images/RNN/12.png)

The `LSTM` can selectively add or remove information from the cell state using **gates**.

![figure 13](images/RNN/13.png)

Each gate consists of a `sigmoid` function and element-wise multiplication to control information flow. The sigmoid output lies in $[0,1]$, where $0$ blocks all information and $1$ allows all information to pass.

##### Gates in LSTM

**Forget gate**

The first step in an `LSTM` is deciding which information to discard from the previous cell state. This decision is made by the forget gate, which takes $h_{t-1}$ and $x_t$ as inputs and outputs values in $[0,1]$ for each element of $C_{t-1}$. A value of $1$ means retaining all information, while $0$ means discarding it.

![figure 14](images/RNN/14.png)
![figure 15](images/RNN/15.png)
![figure 16](images/RNN/16.png)
![figure 17](images/RNN/17.png)


##### Input gate

After determining which information should be forgotten, the next step is deciding **what new information should be stored** in the cell state. This process is controlled by the **input gate**.

The input gate consists of two components:

1. A `sigmoid` layer that determines which values should be updated.
2. A `tanh` layer that generates candidate values to be added to the cell state.

Formally, the input gate is defined as:
$$
\begin{aligned}
\boldsymbol{i}^{(t)} &= \sigma(\boldsymbol{W}_{xi}\boldsymbol{x}^{(t)} + \boldsymbol{W}_{hi}\boldsymbol{h}^{(t-1)} + \boldsymbol{b}_i) \\
\tilde{\boldsymbol{C}}^{(t)} &= \tanh(\boldsymbol{W}_{xC}\boldsymbol{x}^{(t)} + \boldsymbol{W}_{hC}\boldsymbol{h}^{(t-1)} + \boldsymbol{b}_C)
\end{aligned}
$$

Here:
- $\boldsymbol{i}^{(t)}$ controls **which components** of the candidate information should be written.
- $\tilde{\boldsymbol{C}}^{(t)}$ represents the **new candidate content** to be added to the cell state.

The cell state is then updated as follows:
$$
\begin{aligned}
\boldsymbol{C}^{(t)} = \boldsymbol{f}^{(t)} \odot \boldsymbol{C}^{(t-1)} + \boldsymbol{i}^{(t)} \odot \tilde{\boldsymbol{C}}^{(t)}
\end{aligned}
$$

This equation highlights the key strength of the LSTM:  
the cell state is updated through **controlled linear interactions**, which enables stable gradient propagation over long time horizons.

---

##### Output gate

Finally, the **output gate** determines which parts of the cell state should be exposed as the hidden state output at the current time step.

The output gate is defined as:
$$
\begin{aligned}
\boldsymbol{o}^{(t)} &= \sigma(\boldsymbol{W}_{xo}\boldsymbol{x}^{(t)} + \boldsymbol{W}_{ho}\boldsymbol{h}^{(t-1)} + \boldsymbol{b}_o)
\end{aligned}
$$

The hidden state is then computed by applying a `tanh` activation to the cell state and filtering it through the output gate:
$$
\begin{aligned}
\boldsymbol{h}^{(t)} = \boldsymbol{o}^{(t)} \odot \tanh(\boldsymbol{C}^{(t)})
\end{aligned}
$$

This mechanism allows the LSTM to:
- Preserve long-term information in the cell state $\boldsymbol{C}^{(t)}$
- Selectively expose relevant information via the hidden state $\boldsymbol{h}^{(t)}$

---

##### Summary of LSTM gating mechanisms

An LSTM cell regulates information flow using three gates:

1. **Forget gate** — determines what information to discard.
2. **Input gate** — determines what new information to store.
3. **Output gate** — determines what information to reveal to the next layer or time step.

Together, these gates enable LSTMs to effectively model **long-range dependencies**, overcoming the vanishing and exploding gradient problems commonly encountered in standard RNNs.

As a result, LSTMs have become a foundational architecture for sequential modeling tasks such as language modeling, speech recognition, and time-series forecasting.